# 02 Generate Synthetic ELR Data

This notebook creates synthetic ELR surveillance data for analysis.

It generates:

- `elr_data.csv`
- `lab_results.csv`
- `facility.csv`
- `county_district_mapping.csv`

The data intentionally includes:

- missing race
- missing ethnicity
- missing phone/address
- delayed reporting
- reporting spikes
- facility dropouts
- backlog dumps

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
raw_path = Path("../data/raw")
raw_path.mkdir(parents=True, exist_ok=True)

In [4]:
np.random.seed(42)

In [5]:
# Basic facility reference table
facilities = pd.DataFrame({
    "facility_id": ["F001", "F002", "F003", "F004", "F005"],
    "facility_name": [
        "Northside Medical Center",
        "Central County Hospital",
        "Valley Diagnostic Lab",
        "Riverbend Clinic",
        "South Regional Hospital"
    ],
    "clia_id": [
        "11D0001111",
        "22D0002222",
        "33D0003333",
        "44D0004444",
        "55D0005555"
    ],
    "county": [
        "Adams",
        "Baker",
        "Clark",
        "Davis",
        "Evans"
    ],
    "district": [
        "North",
        "North",
        "Central",
        "Central",
        "South"
    ]
})

facilities

,facility_id,facility_name,clia_id,county,district
0,F001,Northside Medical Center,11D0001111,Adams,North
1,F002,Central County Hospital,22D0002222,Baker,North
2,F003,Valley Diagnostic Lab,33D0003333,Clark,Central
3,F004,Riverbend Clinic,44D0004444,Davis,Central
4,F005,South Regional Hospital,55D0005555,Evans,South


In [6]:
records = []

start_date = pd.to_datetime("2025-01-01")
num_days = 90
message_id = 1

race_options = ["White", "Black", "Asian", "Other", "Unknown", "", None]
ethnicity_options = ["Hispanic", "Not Hispanic", "Unknown", "", None]

for day in range(num_days):
    report_date = start_date + pd.Timedelta(days=day)

    for _, facility in facilities.iterrows():

        # Most facilities report a steady number of results each day
        daily_volume = np.random.poisson(lam=25)

        # Simulate one facility not reporting for a few days
        if facility["facility_id"] == "F003" and 40 <= day <= 46:
            daily_volume = 0

        # Simulate one facility sending a large backlog on one day
        if facility["facility_id"] == "F005" and day == 60:
            daily_volume = 180

        for i in range(daily_volume):
            collection_date = report_date - pd.Timedelta(days=np.random.randint(0, 5))
            created_date = collection_date + pd.Timedelta(days=np.random.choice([0, 1, 2, 3, 5, 7]))

            records.append({
                "msg_control_id": f"MSG{message_id:06d}",
                "facility_id": facility["facility_id"],
                "facility_name": facility["facility_name"],
                "clia_id": facility["clia_id"],
                "county": facility["county"],
                "patient_race": np.random.choice(
                    race_options,
                    p=[0.45, 0.20, 0.08, 0.12, 0.07, 0.04, 0.04]
                ),
                "patient_ethnicity": np.random.choice(
                    ethnicity_options,
                    p=[0.20, 0.65, 0.07, 0.04, 0.04]
                ),
                "patient_phone": np.random.choice([
                    f"555-{np.random.randint(100,999)}-{np.random.randint(1000,9999)}",
                    "",
                    "000-000-0000",
                    None
                ], p=[0.86, 0.05, 0.04, 0.05]),
                "patient_address": np.random.choice([
                    f"{np.random.randint(100,9999)} Main St",
                    "",
                    "Unknown",
                    None
                ], p=[0.84, 0.05, 0.05, 0.06]),
                "collection_date": collection_date,
                "created_date": created_date,
                "insert_date": created_date
            })

            message_id += 1

In [7]:
elr_data = pd.DataFrame(records)

elr_data.head()

,msg_control_id,facility_id,facility_name,clia_id,county,patient_race,patient_ethnicity,patient_phone,patient_address,collection_date,created_date,insert_date
0,MSG000001,F001,Northside Medical Center,11D0001111,Adams,Black,Not Hispanic,555-314-5426,8422 Main St,2024-12-30,2025-01-04,2025-01-04
1,MSG000002,F001,Northside Medical Center,11D0001111,Adams,,Hispanic,555-376-2184,6496 Main St,2024-12-31,2025-01-03,2025-01-03
2,MSG000003,F001,Northside Medical Center,11D0001111,Adams,Black,Hispanic,555-575-3747,289 Main St,2025-01-01,2025-01-03,2025-01-03
3,MSG000004,F001,Northside Medical Center,11D0001111,Adams,Black,Not Hispanic,555-230-4556,None,2024-12-30,2025-01-02,2025-01-02
4,MSG000005,F001,Northside Medical Center,11D0001111,Adams,Other,Not Hispanic,555-664-8041,5586 Main St,2024-12-29,2025-01-05,2025-01-05


In [8]:
elr_data.shape

(11238, 12)

In [9]:
#Creating Lab results table
lab_results = elr_data[[
    "msg_control_id",
    "facility_id",
    "collection_date",
    "created_date"
]].copy()

lab_results["test_name"] = np.random.choice(
    ["COVID-19 PCR", "Influenza A", "Influenza B", "RSV"],
    size=len(lab_results)
)

lab_results["result_value"] = np.random.choice(
    ["Detected", "Not Detected"],
    size=len(lab_results),
    p=[0.15, 0.85]
)

lab_results.head()

,msg_control_id,facility_id,collection_date,created_date,test_name,result_value
0,MSG000001,F001,2024-12-30,2025-01-04,Influenza B,Detected
1,MSG000002,F001,2024-12-31,2025-01-03,RSV,Not Detected
2,MSG000003,F001,2025-01-01,2025-01-03,Influenza A,Not Detected
3,MSG000004,F001,2024-12-30,2025-01-02,RSV,Not Detected
4,MSG000005,F001,2024-12-29,2025-01-05,RSV,Not Detected


In [10]:
#county Mapping table
county_district_mapping = facilities[["county", "district"]].drop_duplicates()

county_district_mapping

,county,district
0,Adams,North
1,Baker,North
2,Clark,Central
3,Davis,Central
4,Evans,South


In [11]:
elr_data.to_csv(raw_path / "elr_data.csv", index=False)
lab_results.to_csv(raw_path / "lab_results.csv", index=False)
facilities.to_csv(raw_path / "facility.csv", index=False)
county_district_mapping.to_csv(raw_path / "county_district_mapping.csv", index=False)

In [13]:
missing_values = ["", "Unknown", "None", "nan"]

race_missing_pct = (
    elr_data["patient_race"]
    .astype(str)
    .str.strip()
    .isin(missing_values)
    .mean()
    * 100
)

phone_missing_pct = (
    elr_data["patient_phone"]
    .astype(str)
    .str.strip()
    .isin(missing_values + ["000-000-0000"])
    .mean()
    * 100
)

address_missing_pct = (
    elr_data["patient_address"]
    .astype(str)
    .str.strip()
    .isin(missing_values)
    .mean()
    * 100
)

print(f"Race missing/unknown: {race_missing_pct:.1f}%")
print(f"Phone missing/invalid: {phone_missing_pct:.1f}%")
print(f"Address missing/unknown: {address_missing_pct:.1f}%")

Race missing/unknown: 15.3%
Phone missing/invalid: 14.1%
Address missing/unknown: 15.9%


I created synthetic ELR data to mimic the type of reporting data a public health team might receive from multiple laboratories and facilities. The dataset includes facility identifiers, CLIA IDs, patient demographic fields, collection dates, and report dates.

I intentionally left some race, ethnicity, phone, and address values blank, unknown, or invalid. This lets me measure data completeness from the actual fields instead of creating separate “missing” flags. I also added a few reporting issues, such as a facility dropout and a backlog submission, so the analysis has realistic patterns to detect later.